In [23]:
with open("SRR_Acc_List.txt", "r") as f:
    srr_ids = [line.strip() for line in f if line.strip()]

print("SRR_LIST = [")
for i, srr in enumerate(srr_ids):
    end = ","
    if i == len(srr_ids) - 1:
        end = ""
    
    if i % 4 == 0:
        print("    ", end="")

    print(f'"{srr}"', end=end)

    if (i + 1) % 4 == 0 or i == len(srr_ids) - 1:
        print()
    else:
        print(" ", end="")

print("]")

SRR_LIST = [
    "SRR36692421", "SRR36692422", "SRR36692423", "SRR36692424",
    "SRR36692425", "SRR36692426", "SRR36692427", "SRR36692428",
    "SRR36692429", "SRR36692430", "SRR36692431", "SRR36692432",
    "SRR36692433", "SRR36692434", "SRR36692435", "SRR36692436",
    "SRR36692437", "SRR36692438", "SRR36692439", "SRR36692440",
    "SRR36692441", "SRR36692442", "SRR36692443", "SRR36692444",
    "SRR36692445", "SRR36692446", "SRR36692447", "SRR36692448",
    "SRR36692449", "SRR36692450", "SRR36692451", "SRR36692452",
    "SRR36692453", "SRR36692454", "SRR36692455", "SRR36692456",
    "SRR36692457", "SRR36692458", "SRR36692459", "SRR36692460",
    "SRR36692461", "SRR36692462", "SRR36692463", "SRR36692464",
    "SRR36692465", "SRR36692466", "SRR36692467", "SRR36692468",
    "SRR36692469", "SRR36692470", "SRR36692471", "SRR36692472",
    "SRR36692473", "SRR36692474"
]


In [24]:
import pandas as pd
import glob
import os
from itertools import combinations

# 1. Setup paths - change this to your directory if different
file_pattern = "count_matrix_SRR*.csv"
files = glob.glob(file_pattern)

if len(files) < 2:
    print(f"Not enough files found to compare. Found: {len(files)}")
else:
    print(f"Found {len(files)} files. Starting pair-wise comparison...\n")

# 2. Load all data into a dictionary for speed
# Key: SRR ID (extracted from filename), Value: The DataSeries (Counts)
data_store = {}

for f in files:
    srr_id = os.path.basename(f).replace("count_matrix_", "").replace(".csv", "")
    # Load CSV, use first column (Gene ID) as index
    df = pd.read_csv(f, index_col=0)
    # We store just the first column of data to ignore the header name during comparison
    data_store[srr_id] = df.iloc[:, 0]

# 3. Compare all possible pairs
found_match = False
all_pairs = list(combinations(data_store.keys(), 2))

print(f"{'Comparison Pair':<40} | {'Status'}")
print("-" * 60)

for srr_a, srr_b in all_pairs:
    series_a = data_store[srr_a]
    series_b = data_store[srr_b]
    
    # Check if indices match and values are exactly the same
    # We use .equals() which is robust for pandas objects
    is_match = series_a.equals(series_b)
    
    status = "-  PERFECT MATCH FOUND" if is_match else "No Match"
    print(f"{srr_a} vs {srr_b:<23} | {status}")
    
    if is_match:
        found_match = True

if not found_match:
    print("\nAudit Complete: No duplicate data found across all samples.")
else:
    print("\nAudit Complete: Duplicate data detected. Check the log above.")

Found 51 files. Starting pair-wise comparison...

Comparison Pair                          | Status
------------------------------------------------------------
SRR36692421 vs SRR36692422             | No Match
SRR36692421 vs SRR36692424             | No Match
SRR36692421 vs SRR36692425             | No Match
SRR36692421 vs SRR36692426             | No Match
SRR36692421 vs SRR36692427             | No Match
SRR36692421 vs SRR36692428             | No Match
SRR36692421 vs SRR36692429             | No Match
SRR36692421 vs SRR36692430             | No Match
SRR36692421 vs SRR36692431             | No Match
SRR36692421 vs SRR36692432             | No Match
SRR36692421 vs SRR36692433             | No Match
SRR36692421 vs SRR36692434             | No Match
SRR36692421 vs SRR36692435             | No Match
SRR36692421 vs SRR36692436             | No Match
SRR36692421 vs SRR36692437             | No Match
SRR36692421 vs SRR36692439             | No Match
SRR36692421 vs SRR36692440             

In [25]:
import pandas as pd
import glob
import os

# 1. Configuration
TARGET_ROWS = 62710
file_pattern = "count_matrix_SRR*.csv"
files = glob.glob(file_pattern)

print(f"Auditing {len(files)} files for Row Count Consistency...")
print(f"{'Filename':<30} | {'Rows':<10} | {'Status'}")
print("-" * 55)

mismatch_found = False

for f in files:
    filename = os.path.basename(f)
    # Load CSV - we only need the index to check length (saves memory)
    df = pd.read_csv(f, usecols=[0])
    row_count = len(df)
    
    if row_count != TARGET_ROWS:
        status = f"X MISMATCH (Diff: {row_count - TARGET_ROWS})"
        mismatch_found = True
    else:
        status = "tickV OK"
        
    print(f"{filename:<30} | {row_count:<10} | {status}")

# 2. Final Summary
print("-" * 55)
if mismatch_found:
    print("CRITICAL: Some files have inconsistent dimensions. Do not merge yet.")
else:
    print("SUCCESS: All files are consistent and ready for the Master Merge.")

Auditing 51 files for Row Count Consistency...
Filename                       | Rows       | Status
-------------------------------------------------------
count_matrix_SRR36692421.csv   | 62710      | tickV OK
count_matrix_SRR36692422.csv   | 62710      | tickV OK
count_matrix_SRR36692424.csv   | 62710      | tickV OK
count_matrix_SRR36692425.csv   | 62710      | tickV OK
count_matrix_SRR36692426.csv   | 62710      | tickV OK
count_matrix_SRR36692427.csv   | 62710      | tickV OK
count_matrix_SRR36692428.csv   | 62710      | tickV OK
count_matrix_SRR36692429.csv   | 62710      | tickV OK
count_matrix_SRR36692430.csv   | 62710      | tickV OK
count_matrix_SRR36692431.csv   | 62710      | tickV OK
count_matrix_SRR36692432.csv   | 62710      | tickV OK
count_matrix_SRR36692433.csv   | 62710      | tickV OK
count_matrix_SRR36692434.csv   | 62710      | tickV OK
count_matrix_SRR36692435.csv   | 62710      | tickV OK
count_matrix_SRR36692436.csv   | 62710      | tickV OK
count_matrix_SRR366

In [26]:
import pandas as pd
import glob
import os

# 1. Setup file discovery
file_pattern = "count_matrix_SRR*.csv"
files = sorted(glob.glob(file_pattern))  # sorted ensures SRR IDs stay in order

if not files:
    print("No files found! Check your file path and naming convention.")
else:
    print(f"Found {len(files)} files. Starting the merge...")

# 2. Process and Merge
dfs = []

for f in files:
    # index_col=0 treats the first (unnamed) column as the ID to join on
    df = pd.read_csv(f, index_col=0)
    dfs.append(df)

# 3. Concatenate all dataframes side-by-side
# join='outer' ensures we don't lose genes that might be missing in some samples
master_matrix = pd.concat(dfs, axis=1, join='outer')

# 4. Fill any missing values with 0 (standard for RNA-seq counts)
master_matrix = master_matrix.fillna(0).astype(int)

# 5. Save the final product
output_name = "master_count_matrix.csv"
master_matrix.to_csv(output_name)

print("-" * 30)
print(f"Merge Complete!")
print(f"Final Matrix Shape: {master_matrix.shape[0]} genes x {master_matrix.shape[1]} samples")
print(f"File saved as: {output_name}")

Found 51 files. Starting the merge...
------------------------------
Merge Complete!
Final Matrix Shape: 62710 genes x 51 samples
File saved as: master_count_matrix.csv
